# 🎬 Gerador de Legendas SRT
Transcreve o áudio de um vídeo e salva um arquivo `.srt` usando **OpenAI Whisper**.

> **Dependências:**  `pip install openai-whisper` + `ffmpeg` instalado no sistema.

In [ ]:
# ── 1. Importações ───────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from functions.GenLeg import (
    ConfigLegenda, PERFIS,
    verificar_dependencias, verificar_arquivo,
    carregar_modelo, transcrever,
    formatar_segmentos, salvar_srt,
    exibir_previa, gerar_legenda_srt,
)

print('✅ Funções importadas com sucesso.')

In [ ]:
# ── 2. Parâmetros  ← edite aqui ──────────────────────────────────────────────

ARQUIVO_VIDEO = r"movie_test\BBC.Human.Origins.mkv"  # caminho do vídeo ou áudio
IDIOMA        = "en"     # 'pt', 'en', 'es', 'fr' … ou None = detecção automática
MODELO        = "large"  # tiny | base | small | medium | large
ARQUIVO_SAIDA = None     # None = mesmo nome do vídeo + .srt
DEVICE        = None     # None = detecta automaticamente | 'cuda' | 'cpu'

# ── Configuração de formatação da legenda ─────────────────────────────────────
#
# Perfis prontos — descomente um para usar:
# cfg = PERFIS['padrao']    # 42 chars, 2 linhas, 1–7s  (padrão)
# cfg = PERFIS['cinema']    # 36 chars, 2 linhas, 1.5–6s
# cfg = PERFIS['redes']     # 28 chars, 1 linha,  até 4s, MAIÚSCULAS
# cfg = PERFIS['broadcast'] # 42 chars, 2 linhas, 1–8s
# cfg = PERFIS['acessivel'] # 37 chars, 2 linhas, 2–6s
#
# Ou configure manualmente:
cfg = ConfigLegenda(
    # ── Tempo ──────────────────────────────────────
    duracao_minima   = 1.0,   # duração mínima de um bloco (segundos)
    duracao_maxima   = 7.0,   # duração máxima — blocos maiores são quebrados
    gap_entre_blocos = 0.05,  # silêncio entre blocos consecutivos (segundos)

    # ── Texto ──────────────────────────────────────
    max_chars_por_linha = 42,    # máximo de caracteres por linha
    max_linhas          = 2,     # máximo de linhas por frame (1 ou 2)
    uppercase           = False, # True → TUDO EM MAIÚSCULAS
    remover_pontuacao   = False, # True → remove pontuação no final da linha
)

print('✅ Parâmetros definidos.')

In [ ]:
# ── 3. Verificações ──────────────────────────────────────────────────────────
verificar_dependencias()
verificar_arquivo(ARQUIVO_VIDEO)
print('✅ Tudo certo. Pronto para transcrever.')

In [ ]:
# ── 4. Carrega o modelo ──────────────────────────────────────────────────────
model = carregar_modelo(MODELO, device=DEVICE)

In [ ]:
# ── 5. Transcreve ────────────────────────────────────────────────────────────
segmentos_brutos, idioma_detectado = transcrever(model, ARQUIVO_VIDEO, IDIOMA)

In [ ]:
# ── 6. Formata e salva o .srt ────────────────────────────────────────────────
from pathlib import Path

segmentos = formatar_segmentos(segmentos_brutos, cfg)
print(f'   Segmentos brutos    : {len(segmentos_brutos)}')
print(f'   Segmentos formatados: {len(segmentos)}\n')

saida = ARQUIVO_SAIDA or str(Path(ARQUIVO_VIDEO).with_suffix('.srt'))
caminho_gerado = salvar_srt(segmentos, saida)

In [ ]:
# ── 7. Prévia dos primeiros segmentos ────────────────────────────────────────
exibir_previa(segmentos, n=10)